# CS-4063 — Natural Language Processing | Assignment 3
## Transformer-based Review Understanding with RAG-Enhanced Explanation Generation
**University:** FAST National University of Computer & Emerging Sciences  
**Course:** CS-4063 Natural Language Processing  



### System Overview
```
Amazon Reviews (3 categories, 30k–45k samples)
          │
          ▼
    Preprocessing  (clean → tokenise → vocab → encode → split)
          │
          ▼
  Part A: Encoder-Only Transformer  ──► embeddings saved to disk
          │                                      │
          ▼                                      ▼
  Part B: Retrieval Module  ◄── cosine similarity on saved embeddings
          │
          ▼
  Part C: Decoder-Only Transformer  (RAG input → explanation text)
```

---
## Section 0 — Environment Setup & Global Hyperparameters

All hyperparameters live in one place so they are easy to tune.  
See **Section 8** for the full hyperparameter tuning log.

In [1]:
import os, re, json, gzip, math, random, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{'='*55}")
print(f"  Device        : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU Name      : {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"{'='*55}")

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)
print("\n  models/ and results/ directories ready.")

# ── Hyperparameters ────────────────────────────────────────────────────────
# Encoder
MAX_SEQ_LEN    = 96     # max tokens per review (shorter = much faster)
VOCAB_MIN_FREQ = 4      # prune rare tokens
EMBED_DIM      = 128    # d_model for both encoder and decoder
NUM_HEADS      = 4      # attention heads  (d_k = 32 per head)
NUM_ENC_LAYERS = 2      # encoder depth (2 is fast, 3 gives marginal gains)
FF_DIM         = 256    # FFN hidden size
DROPOUT        = 0.1

# Decoder
DEC_MAX_LEN    = 80     # decoder sequence length
NUM_DEC_LAYERS = 2

# Training
BATCH_SIZE     = 128    # larger batch = faster (use 64 if OOM)
ENC_EPOCHS     = 8      # encoder epochs
DEC_EPOCHS     = 8      # decoder epochs
LR             = 5e-4

# RAG
TOP_K          = 3      # neighbours to retrieve
MAX_GEN_LEN    = 28     # max new tokens at inference

print("\n  Hyperparameters loaded:")
print(f"    MAX_SEQ_LEN={MAX_SEQ_LEN}  EMBED_DIM={EMBED_DIM}  HEADS={NUM_HEADS}")
print(f"    ENC_LAYERS={NUM_ENC_LAYERS}  DEC_LAYERS={NUM_DEC_LAYERS}  FF_DIM={FF_DIM}")
print(f"    BATCH={BATCH_SIZE}  LR={LR}  ENC_EPOCHS={ENC_EPOCHS}  DEC_EPOCHS={DEC_EPOCHS}")
print(f"    TOP_K={TOP_K}  MAX_GEN_LEN={MAX_GEN_LEN}")

  Device        : cpu

  models/ and results/ directories ready.

  Hyperparameters loaded:
    MAX_SEQ_LEN=96  EMBED_DIM=128  HEADS=4
    ENC_LAYERS=2  DEC_LAYERS=2  FF_DIM=256
    BATCH=128  LR=0.0005  ENC_EPOCHS=8  DEC_EPOCHS=8
    TOP_K=3  MAX_GEN_LEN=28


---
## Section 1 — Dataset Loading

We combine three Amazon product categories to ensure domain diversity:

| Category | Domain | Reviews |
|---|---|---|
| **Appliances** | Household appliances (vacuums, refrigerators, etc.) | ≤ 15 000 |
| **Luxury Beauty** | High-end cosmetics, skincare, fragrances | ≤ 15 000 |
| **Video Games** | Gaming consoles, games, peripherals | ≤ 15 000 |

**Total target: 30 000–45 000 samples** as required by the assignment.

Each sample contains:
- `text` — the review body (string)
- `rating` — star rating 1–5 (integer)
- `category` — which product domain (string)

Reviews with fewer than 5 words are discarded as they carry no useful signal.

In [2]:
def load_amazon_gz(filepath, max_samples=15000):
    """
    Stream-load a gzipped Amazon review JSON-lines file.
    We parse line by line to avoid loading the entire file into RAM.
    Only reviews with actual text and a rating are kept.
    """
    records = []
    print(f"  Loading {filepath} ...", end=' ', flush=True)
    t0 = time.time()
    with gzip.open(filepath, 'rt', encoding='utf-8') as fh:
        for line in fh:
            if len(records) >= max_samples:
                break
            try:
                obj    = json.loads(line.strip())
                text   = obj.get('reviewText', '').strip()
                rating = obj.get('overall', None)
                if text and rating is not None and len(text.split()) >= 5:
                    records.append({'text': text, 'rating': int(rating)})
            except (json.JSONDecodeError, KeyError):
                continue
    elapsed = time.time() - t0
    print(f"loaded {len(records):,} reviews in {elapsed:.1f}s")
    return records

# Load all three categories
FILES = {
    'Appliances':    'Appliances.json.gz',
    'Luxury_Beauty': 'Luxury_Beauty.json.gz',
    'Video_Games':   'Video_Games.json.gz',
}

print("Loading Amazon Review datasets...")
print("-" * 50)
all_records = []
for category, fname in FILES.items():
    batch = load_amazon_gz(fname, max_samples=15000)
    for r in batch:
        r['category'] = category
    all_records.extend(batch)

# Shuffle to mix categories evenly
random.shuffle(all_records)

print("-" * 50)
print(f"  Total reviews loaded : {len(all_records):,}")
print()

# Per-category breakdown
cat_counts = Counter(r['category'] for r in all_records)
for cat, cnt in cat_counts.items():
    print(f"    {cat:20s}: {cnt:,} reviews  ({cnt/len(all_records)*100:.1f}%)")

# Rating distribution
print()
print("  Rating distribution across all categories:")
rc = Counter(r['rating'] for r in all_records)
for star in sorted(rc):
    bar = '█' * (rc[star] // 500)
    print(f"    {star}★  {bar}  {rc[star]:,}")

Loading Amazon Review datasets...
--------------------------------------------------
  Loading Appliances.json.gz ... loaded 15,000 reviews in 0.1s
  Loading Luxury_Beauty.json.gz ... loaded 15,000 reviews in 0.1s
  Loading Video_Games.json.gz ... loaded 15,000 reviews in 0.2s
--------------------------------------------------
  Total reviews loaded : 45,000

    Luxury_Beauty       : 15,000 reviews  (33.3%)
    Video_Games         : 15,000 reviews  (33.3%)
    Appliances          : 15,000 reviews  (33.3%)

  Rating distribution across all categories:
    1★  ███████  3,904
    2★  ███  1,965
    3★  ██████  3,330
    4★  █████████████  6,740
    5★  ██████████████████████████████████████████████████████████  29,061


---
## Section 2 — Preprocessing Pipeline

The preprocessing pipeline follows exactly the steps outlined in the assignment brief.

### 2.1 Label Definitions

**Task 1 — Sentiment Classification** (3 classes, primary task)  
This is the main task defined by the assignment.
| Rating | Label | Class ID |
|---|---|---|
| 1–2 stars | Negative | 0 |
| 3 stars | Neutral | 1 |
| 4–5 stars | Positive | 2 |

**Task 2 — Helpfulness Bucket** (3 classes, derived feature — our choice)  
We define "helpfulness" based on a combination of review length and lexical diversity.  
A review with more unique words relative to its total length tends to be more informative.  
*Justification:* This is entirely predictable from text alone (no metadata needed), and it forces the encoder to attend to both content depth and vocabulary richness — a useful inductive bias.
| Condition | Label | Class ID |
|---|---|---|
| Unique word ratio < 0.45 OR word count < 20 | Low helpfulness | 0 |
| Unique word ratio 0.45–0.65 AND word count 20–79 | Medium helpfulness | 1 |
| Unique word ratio > 0.65 AND word count ≥ 80 | High helpfulness | 2 |

### 2.2 Text Cleaning
- Convert to lowercase
- Strip HTML tags
- Remove non-alphanumeric characters (preserve apostrophes for contractions)
- Collapse multiple spaces

### 2.3 Tokenization
Simple whitespace tokenisation after cleaning.  
We intentionally avoid sub-word tokenisation to keep the implementation from-scratch.

### 2.4 Vocabulary Construction
Built exclusively from **training data** to prevent any data leakage.  
Tokens appearing fewer than `VOCAB_MIN_FREQ=4` times are replaced by `<UNK>`.  
Special tokens: `<PAD>=0`, `<UNK>=1`, `<BOS>=2`, `<EOS>=3`.

### 2.5 Padding & Truncation
Fixed sequence length of `MAX_SEQ_LEN=96` tokens.  
Shorter sequences are right-padded with `<PAD>`.  
Longer sequences are truncated from the right.

In [3]:
# ── Step 1: Label assignment ──────────────────────────────────────────────
def rating_to_sentiment(r):
    """3-class sentiment label from star rating."""    
    if r <= 2:  return 0   # Negative
    elif r == 3: return 1  # Neutral
    else:        return 2  # Positive

def compute_helpfulness(text):
    """
    Derived feature: helpfulness bucket based on lexical diversity + length.
    Lexical diversity = unique_words / total_words (type-token ratio).
    High diversity + sufficient length => more informative review.
    """
    words = text.lower().split()
    wc    = len(words)
    if wc == 0: return 0
    diversity = len(set(words)) / wc
    if diversity > 0.65 and wc >= 80:  return 2   # High
    elif diversity < 0.45 or wc < 20:  return 0   # Low
    else:                               return 1   # Medium

print("Assigning labels to all records...")
for r in all_records:
    r['sentiment']    = rating_to_sentiment(r['rating'])
    r['helpfulness']  = compute_helpfulness(r['text'])

# ── Step 2: Train / Validation / Test split (70 / 15 / 15) ───────────────
n       = len(all_records)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

train_data = all_records[:n_train]
val_data   = all_records[n_train : n_train + n_val]
test_data  = all_records[n_train + n_val :]

print(f"\nDataset split (70/15/15):")
print(f"  Training   : {len(train_data):,} samples")
print(f"  Validation : {len(val_data):,} samples")
print(f"  Test       : {len(test_data):,} samples")
print(f"  Total      : {len(all_records):,} samples")

print("\nSentiment distribution per split:")
SENT_NAMES = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
HELP_NAMES = {0: 'Low', 1: 'Medium', 2: 'High'}
for split_name, split in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    sc = Counter(r['sentiment'] for r in split)
    print(f"  {split_name:6s}  →  Neg:{sc[0]:,}  Neu:{sc[1]:,}  Pos:{sc[2]:,}")

Assigning labels to all records...

Dataset split (70/15/15):
  Training   : 31,499 samples
  Validation : 6,750 samples
  Test       : 6,751 samples
  Total      : 45,000 samples

Sentiment distribution per split:
  Train   →  Neg:4,170  Neu:2,307  Pos:25,022
  Val     →  Neg:830  Neu:540  Pos:5,380
  Test    →  Neg:869  Neu:483  Pos:5,399


In [4]:
# ── Step 3: Text cleaning ────────────────────────────────────────────────
def clean_text(raw):
    """
    Full cleaning pipeline:
    1. Lowercase
    2. Strip HTML tags
    3. Keep only letters, digits, apostrophes, and spaces
    4. Collapse whitespace
    """
    text = raw.lower()
    text = re.sub(r'<[^>]+>', ' ', text)          # remove HTML
    text = re.sub(r"[^a-z0-9'\s]", ' ', text)    # keep safe chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenise(text):
    """Clean then split on whitespace."""    
    return clean_text(text).split()

# Verify cleaning works
sample_raw = train_data[0]['text']
sample_tokens = tokenise(sample_raw)
print("Cleaning verification:")
print(f"  Raw (60 chars)    : {sample_raw[:60]}")
print(f"  Cleaned tokens    : {sample_tokens[:12]}")
print(f"  Token count       : {len(sample_tokens)}")

Cleaning verification:
  Raw (60 chars)    : this is fake, don't buy!
  Cleaned tokens    : ['this', 'is', 'fake', "don't", 'buy']
  Token count       : 5


In [5]:
# ── Step 4: Vocabulary construction (training data ONLY) ─────────────────
print("Building vocabulary from training data only...")
PAD_TOK, UNK_TOK, BOS_TOK, EOS_TOK = '<PAD>', '<UNK>', '<BOS>', '<EOS>'

freq = Counter()
for r in train_data:
    freq.update(tokenise(r['text']))

print(f"  Raw token types in training set : {len(freq):,}")

specials   = [PAD_TOK, UNK_TOK, BOS_TOK, EOS_TOK]
kept_words = [w for w, c in freq.most_common() if c >= VOCAB_MIN_FREQ]
all_tokens = specials + kept_words

vocab     = {tok: idx for idx, tok in enumerate(all_tokens)}
inv_vocab = {idx: tok for tok, idx in vocab.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab[PAD_TOK]
UNK_IDX    = vocab[UNK_TOK]
BOS_IDX    = vocab[BOS_TOK]
EOS_IDX    = vocab[EOS_TOK]

# OOV analysis
val_tokens  = sum(len(tokenise(r['text'])) for r in val_data)
val_oov     = sum(1 for r in val_data for t in tokenise(r['text']) if t not in vocab)
test_tokens = sum(len(tokenise(r['text'])) for r in test_data)
test_oov    = sum(1 for r in test_data for t in tokenise(r['text']) if t not in test_data)

print(f"  Vocabulary size (after pruning) : {VOCAB_SIZE:,}  (min_freq={VOCAB_MIN_FREQ})")
print(f"  Val  OOV rate                   : {val_oov/max(val_tokens,1):.2%}")
print(f"  Most common tokens              : {kept_words[:10]}")

# Save vocabulary
with open('results/vocab.pkl', 'wb') as f:
    pickle.dump({'vocab': vocab, 'inv_vocab': inv_vocab,
                 'vocab_size': VOCAB_SIZE, 'min_freq': VOCAB_MIN_FREQ}, f)
print("\n  Vocabulary saved → results/vocab.pkl")

Building vocabulary from training data only...
  Raw token types in training set : 37,121
  Vocabulary size (after pruning) : 12,975  (min_freq=4)
  Val  OOV rate                   : 2.09%
  Most common tokens              : ['the', 'and', 'i', 'a', 'to', 'it', 'is', 'this', 'of', 'you']

  Vocabulary saved → results/vocab.pkl
